In [ ]:
import os
import zipfile
import json
import random
import time
import io
import math
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.metrics import fbeta_score

# -------------------------------------------------------------------
# 1. ACADEMIC CONFIGURATION
# -------------------------------------------------------------------

# Google Drive Paths
DRIVE_ROOT = "/content/drive/MyDrive/pilskalnu_projekts"
ZIP_FILE_PATH = os.path.join(DRIVE_ROOT, "algoritma_datu_kopa.zip")
CACHE_FILE_PATH = os.path.join(DRIVE_ROOT, "dataset_index_v3_structured.json")
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints_v2")

# Hardware Settings
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2
PIN_MEMORY = True

# Hyperparameters
IMG_SIZE = 768
BATCH_SIZE = 16
INPUT_CHANNELS = 6
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2

# Training Dynamics
PATIENCE_BUFFER = 100
UNFREEZE_THRESHOLD = 0.005

# -------------------------------------------------------------------
# 2. ROBUST DATASET INDEXING
# -------------------------------------------------------------------

def index_zip_dataset(zip_path, cache_path):
    if os.path.exists(cache_path):
        print(f"- [CACHE] Loading index from: {os.path.basename(cache_path)}")
        with open(cache_path, "r") as f:
            data = json.load(f)
            return data["hillforts"], data["regulars"]

    print(f"- [INDEXING] Scanning Zip file structure...")

    relief_paths = []
    all_paths_set = set()

    with zipfile.ZipFile(zip_path, "r") as zf:
        all_names = zf.namelist()

    for name in all_names:
        if name.startswith("__MACOSX") or name.startswith("."): continue
        if not name.endswith((".png", ".jpg", ".tif")): continue

        clean_name = name.replace("\\", "/")
        all_paths_set.add(clean_name)

        if "reljefs" in clean_name.lower():
            relief_paths.append(clean_name)

    hillforts, regulars = [], []

    for r_path in relief_paths:
        s_path_candidate = r_path.replace("reljefs", "slipums")

        if s_path_candidate in all_paths_set:
            pair = (r_path, s_path_candidate)
            if "pilskalni" in r_path.lower() or "pilskalns" in r_path.lower():
                hillforts.append(pair)
            else:
                regulars.append(pair)

    print(f"- [STATS] Found {len(hillforts)} Hillforts and {len(regulars)} Regular Territories.")

    with open(cache_path, "w") as f:
        json.dump({"hillforts": hillforts, "regulars": regulars}, f)

    return hillforts, regulars

# -------------------------------------------------------------------
# 3. DATASET LOADER
# -------------------------------------------------------------------

class SixChannelDataset(Dataset):
    def __init__(self, zip_path, paired_files, labels, transform=None):
        self.zip_path = zip_path
        self.paired_files = paired_files
        self.labels = labels
        self.transform = transform
        self.zip_handle = None

    def __len__(self):
        return len(self.paired_files)

    def __getitem__(self, idx):
        if self.zip_handle is None:
            self.zip_handle = zipfile.ZipFile(self.zip_path, "r")

        r_path, s_path = self.paired_files[idx]
        label = self.labels[idx]

        try:
            r_bytes = self.zip_handle.read(r_path)
            s_bytes = self.zip_handle.read(s_path)

            img_r = Image.open(io.BytesIO(r_bytes)).convert("RGB")
            img_s = Image.open(io.BytesIO(s_bytes)).convert("RGB")

            if img_r.size != (IMG_SIZE, IMG_SIZE):
                img_r = img_r.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
                img_s = img_s.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

            if self.transform:
                t_r, t_s = self.transform(img_r, img_s)
            else:
                t_r = TF.to_tensor(img_r)
                t_s = TF.to_tensor(img_s)

            combined = torch.cat((t_r, t_s), dim=0)
            return combined, torch.tensor(label, dtype=torch.float32)

        except Exception as e:
            print(f"Error loading {r_path}: {e}")
            return self.__getitem__(random.randint(0, len(self.paired_files)-1))

# -------------------------------------------------------------------
# 4. TRANSFORMS
# -------------------------------------------------------------------

class DualAugment:
    def __init__(self):
        self.color = T.ColorJitter(brightness=0.2, contrast=0.2)
        self.eraser = T.RandomErasing(p=0.2, scale=(0.02, 0.1))

    def __call__(self, img1, img2):
        if random.random() > 0.5:
            img1, img2 = TF.hflip(img1), TF.hflip(img2)
        if random.random() > 0.5:
            img1, img2 = TF.vflip(img1), TF.vflip(img2)

        angle = random.choice([0, 90, 180, 270])
        if angle != 0:
            img1, img2 = TF.rotate(img1, angle), TF.rotate(img2, angle)

        img1 = self.color(img1)
        img2 = self.color(img2)

        t1, t2 = TF.to_tensor(img1), TF.to_tensor(img2)

        norm = T.Normalize([0.5]*3, [0.5]*3)
        t1, t2 = norm(t1), norm(t2)

        if random.random() < 0.2:
            t1, t2 = self.eraser(t1), self.eraser(t2)

        return t1, t2

# -------------------------------------------------------------------
# 5. GLOBAL CYCLE SAMPLER
# -------------------------------------------------------------------

class GlobalCycleSampler(Sampler):
    def __init__(self, hf_indices, reg_indices, batch_size):
        self.hf_indices = hf_indices
        self.reg_indices = reg_indices
        self.batch_size = batch_size
        self.half_batch = batch_size // 2
        self.global_reg_pool = list(reg_indices)
        random.shuffle(self.global_reg_pool)
        self.reg_pointer = 0
        self.cycle_count = 0

    def __len__(self):
        return math.ceil(len(self.hf_indices) / self.half_batch)

    def __iter__(self):
        random.shuffle(self.hf_indices)
        num_batches = len(self)

        for i in range(num_batches):
            batch = []
            start_h = i * self.half_batch
            end_h = min((i + 1) * self.half_batch, len(self.hf_indices))
            batch.extend(self.hf_indices[start_h:end_h])

            while len(batch) < self.half_batch:
                batch.append(random.choice(self.hf_indices))

            needed = self.half_batch
            regs_to_add = []
            while needed > 0:
                remaining = len(self.global_reg_pool) - self.reg_pointer
                if remaining >= needed:
                    chunk = self.global_reg_pool[self.reg_pointer : self.reg_pointer + needed]
                    regs_to_add.extend(chunk)
                    self.reg_pointer += needed
                    needed = 0
                else:
                    chunk = self.global_reg_pool[self.reg_pointer:]
                    regs_to_add.extend(chunk)
                    needed -= len(chunk)
                    random.shuffle(self.global_reg_pool)
                    self.reg_pointer = 0
                    self.cycle_count += 1
                    print(f"\n[INFO] Global Dataset Cycle Completed! Count: {self.cycle_count}")

            batch.extend(regs_to_add)
            random.shuffle(batch)
            yield batch

# -------------------------------------------------------------------
# 6. MODEL SETUP
# -------------------------------------------------------------------

class HillfortNetV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        old_weights = self.base.conv1.weight
        self.base.conv1 = nn.Conv2d(6, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            self.base.conv1.weight[:, :3] = old_weights
            self.base.conv1.weight[:, 3:] = old_weights
        self.base.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(self.base.fc.in_features, 1)
        )

    def forward(self, x):
        return self.base(x)

# -------------------------------------------------------------------
# 7. TRAINING LOOP (With Checkpoint Loading)
# -------------------------------------------------------------------

def set_parameter_freeze(model, is_frozen):
    if is_frozen:
        for name, p in model.base.named_parameters():
            if "fc" not in name and "conv1" not in name:
                p.requires_grad = False
        print("- [SYSTEM] Backbone FROZEN. Training Heads & Conv1 adapter only.")
    else:
        for p in model.parameters():
            p.requires_grad = True
        print("- [SYSTEM] Backbone UNFROZEN. Full Deep Learning enabled.")

def main():
    if not os.path.exists(DRIVE_ROOT):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except: pass

    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # 1. Prepare Data
    hf_files, reg_files = index_zip_dataset(ZIP_FILE_PATH, CACHE_FILE_PATH)
    all_files = hf_files + reg_files
    all_labels = [1]*len(hf_files) + [0]*len(reg_files)
    hf_indices = list(range(len(hf_files)))
    reg_indices = list(range(len(hf_files), len(all_files)))

    dataset = SixChannelDataset(ZIP_FILE_PATH, all_files, all_labels, transform=DualAugment())
    sampler = GlobalCycleSampler(hf_indices, reg_indices, BATCH_SIZE)
    loader = DataLoader(dataset, batch_sampler=sampler, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    # 2. Setup Model & Loss
    model = HillfortNetV2().to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()

    # 3. Stats & Resume Logic
    batches_per_epoch = len(sampler)
    regular_patches_per_epoch = batches_per_epoch * (BATCH_SIZE // 2)
    total_regulars = len(reg_files)
    epochs_per_cycle = math.ceil(total_regulars / regular_patches_per_epoch) if regular_patches_per_epoch > 0 else 1

    best_score = 0.0
    ema_score = 0.0
    start_epoch = 0
    current_cycle = 0
    patience_counter = 0
    patience_active = False

    # State tracking
    is_frozen = True
    prev_epoch_loss = float('inf')

    # =========================================================
    # CHECKPOINT LOADING LOGIC
    # =========================================================
    checkpoint_path = os.path.join(CHECKPOINT_DIR, "latest.pth")

    if os.path.exists(checkpoint_path):
        print(f"\n[RESUME] Found checkpoint: {checkpoint_path}")
        try:
            checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
            model.load_state_dict(checkpoint['model'])
            optimizer.load_state_dict(checkpoint['optimizer'])
            start_epoch = checkpoint['epoch']
            ema_score = checkpoint.get('ema_f2', 0.0)
            print(f"[RESUME] Resumed from Epoch {start_epoch} | EMA F2: {ema_score:.4f}")
        except Exception as e:
            print(f"[RESUME ERROR] Could not load checkpoint: {e}")
            print("[RESUME] Starting from scratch.")
    else:
        print("\n[START] No checkpoint found. Starting fresh.")

    # Apply Frozen State correctly based on Resume
    # If we are resuming late (e.g., after unfreeze happened), we should unfreeze.
    if start_epoch > 5:
        is_frozen = False
        set_parameter_freeze(model, False)
    else:
        is_frozen = True
        set_parameter_freeze(model, True)

    print(f"\n{'-'*50}")
    print(f"ACADEMIC TRAINING PROTOCOL V2.1 (Resumable)")
    print(f"{'-'*50}")

    # 4. Training Loop
    epoch = start_epoch
    while True:
        epoch += 1
        model.train()

        batch_loss = []
        preds_all, targets_all = [], []
        start_time = time.time()

        for batch_i, (inputs, labels) in enumerate(loader):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            batch_loss.append(loss.item())
            preds = (torch.sigmoid(outputs) > 0.5).float()
            preds_all.extend(preds.cpu().numpy())
            targets_all.extend(labels.cpu().numpy())

            if batch_i % 10 == 0:
                print(f"Epoch {epoch} [{batch_i}/{batches_per_epoch}]", end="\r")

        # Metrics
        avg_loss = np.mean(batch_loss)
        f2 = fbeta_score(targets_all, preds_all, beta=2, zero_division=0)
        ema_score = 0.9 * ema_score + 0.1 * f2 if epoch > 1 else f2

        # Dynamic Unfreeze Logic
        if is_frozen and epoch > 1:
            improvement = prev_epoch_loss - avg_loss
            print(f"   [MONITOR] Loss Improvement: {improvement:.5f}")

            if improvement < UNFREEZE_THRESHOLD:
                print(f"\n   >>> PLATEAU DETECTED. Unfreezing Backbone.")
                set_parameter_freeze(model, False)
                is_frozen = False

        prev_epoch_loss = avg_loss

        time_elapsed = (time.time() - start_time) / 60
        print(f"Epoch {epoch} | Time: {time_elapsed:.1f}m | Loss: {avg_loss:.4f} | F2(Smooth): {ema_score:.4f}")

        # Saving
        state = {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "ema_f2": ema_score
        }
        torch.save(state, os.path.join(CHECKPOINT_DIR, "latest.pth"))

        if ema_score > best_score:
            best_score = ema_score
            torch.save(state, os.path.join(CHECKPOINT_DIR, "best_model_v2.pth"))
            patience_counter = 0
            print(f"   >>> NEW BEST MODEL (Smooth F2: {ema_score:.4f})")

        if sampler.cycle_count > current_cycle:
            current_cycle = sampler.cycle_count
            print(f"\n[SAVE] Full Dataset Cycle {current_cycle} Completed.")
            torch.save(state, os.path.join(CHECKPOINT_DIR, f"cycle_{current_cycle}_finished.pth"))
            patience_active = True

        if patience_active:
            if ema_score < best_score:
                patience_counter += 1
                print(f"   [PATIENCE] {patience_counter}/{PATIENCE_BUFFER} (Active)")

            if patience_counter >= PATIENCE_BUFFER:
                print(f"\n[STOP] Training Finished.")
                break

if __name__ == "__main__":
    main()